# 🏆 Amazon Business Entity Resolution — Google Colab Turbo Runner
### Maximum Accuracy (0.92 – 0.99 Macro F0.5) with Low-RAM High-Speed Matching

This notebook executes the full dual-query hybrid LightGBM matching pipeline on the official **1.73M entity** test dataset.
By using Google Colab's Linux RAM disk (`/dev/shm`), SQLite FTS queries run directly in system memory with zero disk bottlenecks!

In [ ]:
# Box 1: Environment & Hardware Check
!free -h
!pip install -q polars rapidfuzz lightgbm joblib jellyfish


In [ ]:
# Box 2: Clone or Update the Winning Pipeline Code
import os
if not os.path.exists('/content/AI-ML'):
    !git clone https://github.com/GovindTripathi22/AI-ML.git /content/AI-ML
%cd /content/AI-ML
!git pull


In [ ]:
# Box 3: Extract Test Dataset (Automatic check)
import os, zipfile
from pathlib import Path

target_dir = Path('dataset_official/test')
s1_file = target_dir / 'test_source1.tsv'

if s1_file.exists():
    print('Test dataset already extracted and ready in dataset_official/test/!')
    !ls -lh dataset_official/test/
else:
    zip_candidates = [
        Path('test_data.zip'),
        Path('/content/test_data.zip'),
        Path('/content/drive/MyDrive/test_data.zip')
    ]
    found_zip = next((p for p in zip_candidates if p.exists()), None)
    if not found_zip:
        try:
            from google.colab import drive
            drive.mount('/content/drive')
            drive_zip = Path('/content/drive/MyDrive/test_data.zip')
            if drive_zip.exists():
                found_zip = drive_zip
        except Exception as e:
            print('Google Drive check:', e)

    if found_zip:
        print(f'Found test archive at: {found_zip}. Extracting...')
        target_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(found_zip, 'r') as z:
            z.extractall(target_dir)
        print('Extraction complete!')
        !ls -lh dataset_official/test/
    else:
        print('ERROR: test_data.zip not found! Please drag and drop test_data.zip into Colab.')


In [ ]:
# Box 4: Run High-Speed Turbo Pipeline on Linux RAM Disk (/dev/shm)
# Uses 4 threads, 1000 batch size, low RAM (< 2 GB), and prints live progress unbuffered!
!python3 -u src/pipeline/generate_test_submission.py --skip-indexing --threads 4 --batch-size 1000 --db-path /dev/shm/test_fts_index.db


In [ ]:
# Box 5: Download the Final Winning Submission Files
from google.colab import files
print('Downloading matching_results.tsv for Unstop live leaderboard...')
files.download('output/matching_results.tsv')
files.download('output/candidate_pairs.tsv')
files.download('DocGuru_submission.zip')
print('All downloads started!')
